In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.seasonal import STL
from statsmodels.tsa.stattools import adfuller, kpss

DATA_DIR = "/kaggle/input/store-sales-time-series-forecasting"
pd.set_option("display.max_columns", 50)

In [ ]:
train = pd.read_csv(f"{DATA_DIR}/train.csv", parse_dates=["date"])
stores = pd.read_csv(f"{DATA_DIR}/stores.csv")
oil = pd.read_csv(f"{DATA_DIR}/oil.csv", parse_dates=["date"])
holidays = pd.read_csv(f"{DATA_DIR}/holidays_events.csv", parse_dates=["date"])

print(train.shape, stores.shape, oil.shape, holidays.shape)
print(stores["type"].value_counts())
print(sorted(train["family"].unique()))

In [ ]:
def first_sustained_activation(g, date_col="date", value_col="sales", min_active_days=20, window=30):
    g = g.sort_values(date_col).reset_index(drop=True)
    active = (g[value_col] > 0).astype(int)
    forward_active_count = active[::-1].rolling(window, min_periods=1).sum()[::-1].reset_index(drop=True)
    sustained_mask = (active == 1) & (forward_active_count >= min_active_days)
    if not sustained_mask.any():
        return g.loc[active.idxmax(), date_col] if active.any() else g[date_col].iloc[0]
    return g.loc[sustained_mask.idxmax(), date_col]

overall_start = train["date"].min()

store_daily_totals = train.groupby(["store_nbr", "date"])["sales"].sum().reset_index()
store_activation_date = store_daily_totals.groupby("store_nbr", group_keys=False).apply(
    lambda g: first_sustained_activation(g, value_col="sales")
)
store_activation_lag_days = (store_activation_date - overall_start).dt.days

LATE_OPENING_THRESHOLD_DAYS = 400

print(store_activation_lag_days.sort_values(ascending=False).head(10))

In [ ]:
stores_sorted = stores.sort_values(["type", "cluster", "store_nbr"])

def select_stores_with_activation_check(stores_sorted, activation_lag, n_per_type=2, threshold=LATE_OPENING_THRESHOLD_DAYS):
    selected = []
    excluded = []
    for store_type, group in stores_sorted.groupby("type", sort=False):
        picked = []
        for store_nbr in group["store_nbr"].tolist():
            if len(picked) == n_per_type:
                break
            lag = activation_lag.get(store_nbr, np.inf)
            if lag > threshold:
                excluded.append((store_nbr, store_type, lag))
                continue
            picked.append(store_nbr)
        if len(picked) < n_per_type:
            print(f"WARNING: type {store_type} yielded only {len(picked)} stores under the activation threshold")
        selected.extend(picked)
    return selected, excluded

selected_stores, excluded_stores = select_stores_with_activation_check(stores_sorted, store_activation_lag_days)

print("Selected stores:", selected_stores)
if excluded_stores:
    print("Excluded (late-opening) candidates:", excluded_stores)
print(stores[stores["store_nbr"].isin(selected_stores)][["store_nbr", "city", "type", "cluster"]])

In [ ]:
selected_families = ["GROCERY I", "BEVERAGES", "PRODUCE", "CLEANING", "HOME CARE", "DAIRY"]
assert all(f in train["family"].unique() for f in selected_families), "Family name mismatch - check exact spelling in train['family'].unique()"

In [ ]:
subset = train[
    train["store_nbr"].isin(selected_stores) & train["family"].isin(selected_families)
].copy()

subset = subset.merge(stores, on="store_nbr", how="left")
subset = subset.merge(oil, on="date", how="left")
subset["dcoilwtico"] = subset["dcoilwtico"].ffill().bfill()

non_holiday_types = ["Work Day", "Event"]
national_holidays = holidays[
    (holidays["locale"] == "National")
    & (~holidays["transferred"])
    & (~holidays["type"].isin(non_holiday_types))
]["date"].unique()
subset["is_holiday"] = subset["date"].isin(national_holidays).astype(int)

print(subset.shape)
print(subset.isna().sum())

In [ ]:
sample_combo = subset[
    (subset["store_nbr"] == selected_stores[0]) & (subset["family"] == "GROCERY I")
].sort_values("date")

plt.figure(figsize=(14, 4))
plt.plot(sample_combo["date"], sample_combo["sales"])
plt.title(f"Sales — Store {selected_stores[0]}, GROCERY I")
plt.xlabel("Date")
plt.ylabel("Sales")
plt.show()

In [ ]:
def pick_representative_store(subset, family, selected_stores, by="total_sales"):
    family_df = subset[(subset["family"] == family) & (subset["store_nbr"].isin(selected_stores))]
    if by == "total_sales":
        scores = family_df.groupby("store_nbr")["sales"].sum()
    elif by == "sales_cv":
        grouped = family_df.groupby("store_nbr")["sales"]
        scores = grouped.std() / grouped.mean().replace(0, np.nan)
    elif by == "mean_promo":
        scores = family_df.groupby("store_nbr")["onpromotion"].mean()
    return int(scores.idxmax())

representative_combos = [
    (pick_representative_store(subset, "GROCERY I", selected_stores, by="total_sales"), "GROCERY I"),
    (pick_representative_store(subset, "PRODUCE", selected_stores, by="sales_cv"), "PRODUCE"),
    (pick_representative_store(subset, "BEVERAGES", selected_stores, by="mean_promo"), "BEVERAGES"),
]
print("Representative combos:", representative_combos)

for store_nbr, family in representative_combos:
    series = subset[
        (subset["store_nbr"] == store_nbr) & (subset["family"] == family)
    ].sort_values("date").set_index("date")["sales"]
    series = series.asfreq("D").fillna(0)

    stl = STL(series, period=7, robust=True)
    result = stl.fit()

    fig = result.plot()
    fig.suptitle(f"STL Decomposition — Store {store_nbr}, {family}", y=1.02)
    plt.show()

In [ ]:
def stationarity_tests(series):
    series = series.dropna()
    adf_stat, adf_p, *_ = adfuller(series)
    try:
        kpss_stat, kpss_p, *_ = kpss(series, regression="c", nlags="auto")
    except Exception:
        kpss_stat, kpss_p = np.nan, np.nan
    return adf_p, kpss_p

results = []
for store_nbr in selected_stores:
    for family in selected_families:
        series = subset[
            (subset["store_nbr"] == store_nbr) & (subset["family"] == family)
        ].sort_values("date").set_index("date")["sales"].asfreq("D").fillna(0)
        adf_p, kpss_p = stationarity_tests(series)
        results.append({
            "store_nbr": store_nbr, "family": family,
            "adf_p_value": adf_p, "kpss_p_value": kpss_p,
            "likely_stationary_adf": adf_p < 0.05,
        })

stationarity_df = pd.DataFrame(results)
print(stationarity_df)
print(f"\n% series stationary by ADF: {stationarity_df['likely_stationary_adf'].mean()*100:.1f}%")

In [ ]:
series_activation = subset.groupby(["store_nbr", "family"]).apply(
    lambda g: first_sustained_activation(g, value_col="sales")
)
series_activation = series_activation.rename("activation_date").reset_index()

all_combos = pd.MultiIndex.from_product(
    [selected_stores, selected_families], names=["store_nbr", "family"]
)
series_activation = (
    series_activation.set_index(["store_nbr", "family"])
    .reindex(all_combos)
    .reset_index()
)
series_activation["activation_date"] = series_activation["activation_date"].fillna(subset["date"].min())

print(series_activation.sort_values("activation_date", ascending=False).head(10))

In [ ]:
subset.to_parquet("/kaggle/working/retailcast_subset.parquet", index=False)
stationarity_df.to_csv("/kaggle/working/stationarity_results.csv", index=False)
series_activation.to_csv("/kaggle/working/series_activation_diagnostics.csv", index=False)

import json
with open("/kaggle/working/subset_config.json", "w") as f:
    json.dump({
        "selected_stores": selected_stores,
        "selected_families": selected_families,
        "representative_combos": representative_combos,
    }, f, indent=2)

print("Saved: retailcast_subset.parquet, stationarity_results.csv, series_activation_diagnostics.csv, subset_config.json")